# Lab 5 : Store the vectors, then search

*W3 RAG Part 1 · Utrains LLMOps 8-Week Course*

Run each cell in order. Read the output. Move to the next.

See the matching slide in this week's concepts deck for the real-world story this lab teaches.


## What we are achieving in this lab

**Objective.** Put Lab 4's chunk vectors in a **vector store**. Search with `retriever.invoke`. See that it is the same cosine ranking, not a new algorithm.

**Prerequisites.** Labs 1–4 finished. Same OpenAI key. Same `handbook.txt`.

**The line so far.**

| Lab | What you can do |
|-----|-----------------|
| 1 | Turn a sentence into a vector |
| 2 | Score two vectors with cosine |
| 3 | Search short snippets |
| 4 | Split a file, then embed, then cosine **in a Python list** |
| 5 (this lab) | Keep those vectors in a store. Search without the `for` loop. |

**What this lab uses.**

| Layer | What it does | What we use |
|-------|----------------|-------------|
| Split | Cut the handbook | `RecursiveCharacterTextSplitter` (Lab 4, size 500) |
| Embed | Chunk → vector | OpenAI `text-embedding-3-small` |
| Store | Keep vectors, return nearest | LangChain `InMemoryVectorStore` |

**What you will do.**

1. Split `handbook.txt` the Lab 4 way (500 / 50). Print the chunks.
2. Rank them with the Lab 4 cosine loop.
3. Store the same chunks. Print how many vectors went in.
4. Search with `retriever.invoke`. Check the top hit matches the loop.

**Cost.** A few embedding calls. Fractions of a cent.


## Where this sits after Lab 4

Lab 4: split, embed every chunk, cosine against the question, pick the highest. That loop is the truth.

A real product will not score every chunk in Python on every question. A **vector store** keeps the vectors and, at question time, returns the closest few.

`InMemoryVectorStore` does Labs 1 and 2 for you:

- **index** (once): `embed_documents` on each chunk, keep the vectors
- **search** (every question): `embed_query`, then cosine, return top `k`

It lives only in this kernel. Restart, and it is gone. That is fine for a lab. Production stores (named after you have seen this one work) keep the vectors on disk or in a database.

No chat model today. Search only. Lab 6 is when those hits go into a prompt.


### Step 1. Same key, same handbook, Lab 4 split

If the key cell fails, fix `.env` from Lab 1 first.


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "Missing OPENAI_API_KEY. Copy .env.example to .env at the repository root, "
        "paste the key, and restart the kernel."
    )

print("OPENAI_API_KEY : set")


In [ ]:
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter

candidates = [Path("handbook.txt"), Path("week03") / "handbook.txt"]
path = None
for p in candidates:
    if p.exists():
        path = p
        break
if path is None:
    raise FileNotFoundError(
        "handbook.txt not found. Open this notebook from week03/, "
        "or start Jupyter from the repository root."
    )

HANDBOOK = path.read_text(encoding="utf-8")

# Lab 4: 500 was the usable size for this file.
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(HANDBOOK)

print("file   :", path)
print("chunks :", len(chunks), "  (chunk_size=500, overlap=50)")
print()
for i, text in enumerate(chunks):
    print("--- chunk", i, " (", len(text), "chars) ---")
    print(text)
    print()


### Step 2. Same embeddings and cosine as Labs 1–2


In [ ]:
import numpy as np
from langchain_openai import OpenAIEmbeddings

EMBED_MODEL = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)


def embed(text: str) -> np.ndarray:
    # Lab 1: one string -> one vector.
    return np.asarray(embeddings.embed_query(text), dtype=float)


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    # Lab 2: how close two arrows are. Higher = closer in meaning.
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


QUERY = "How do I get reimbursed for a $300 train ticket?"
print("Question:", QUERY)
print("Model   :", EMBED_MODEL)


### Step 3. Lab 4 ranking, as a baseline

Same cosine loop. Read the order. Step 4 should return the same top idea.


In [ ]:
print("Question:", QUERY)
print()

vecs = embeddings.embed_documents(chunks)
q_vec = embed(QUERY)

scored = []
for i, text in enumerate(chunks):
    score = cosine(q_vec, np.asarray(vecs[i], dtype=float))
    scored.append((score, i, text))
    preview = text.replace("\n", " ")
    if len(preview) > 90:
        preview = preview[:90] + "..."
    print(round(score, 3), "  chunk", i, " ", preview)

print()
scored.sort(reverse=True)
print("Highest: chunk", scored[0][1], "  cosine", round(scored[0][0], 3))
print()
print(scored[0][2])


### Step 4. Put the same chunks in a vector store

`Document` is a wrapper: the text plus a chunk number. That number is how you later say "this came from chunk 2," not a new kind of search.

`from_documents` embeds every chunk once and stores the vectors. That is the **index**. It runs when the handbook changes, not on every question.


In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

docs = []
for i, text in enumerate(chunks):
    docs.append(Document(page_content=text, metadata={"chunk": i}))

vectorstore = InMemoryVectorStore.from_documents(docs, embedding=embeddings)

print("stored", len(docs), "chunks in InMemoryVectorStore")
print("model :", EMBED_MODEL)
print()
print("This store lives in this kernel only. Restart the kernel and it is empty.")


### Step 5. Search the store

`k=3` means "give me the three closest chunks."

`retriever.invoke(question)` is Lab 2 packaged: `embed_query`, then cosine, then the top hits.


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Question:", QUERY)
print()
hits = retriever.invoke(QUERY)
for i, hit in enumerate(hits, start=1):
    print("--- hit", i, "  chunk", hit.metadata.get("chunk"), " ---")
    print(hit.page_content)
    print()


Hit 1 should be the same reimbursement chunk your cosine loop put first. The store did not invent a new score. It ran Lab 2 for you.

## Stores you should be able to name (implement one)

You used **InMemoryVectorStore**. Job conversations still use the other names.

| Store | Where the vectors live | When teams pick it |
|-------|------------------------|--------------------|
| **InMemoryVectorStore** (this lab) | This Python process | Labs and tests. Gone on restart. |
| **Chroma** | Files on disk | Small apps, local prototypes. |
| **pgvector** | Postgres | The team already has Postgres. |
| **Pinecone / Qdrant / Weaviate** | Hosted service | Managed scale. You still embed with OpenAI (or Voyage). |
| **OpenSearch** | Search cluster | Keyword (BM25) and vectors in one place. Hybrid later. |

The line you write stays `retriever.invoke(question)` if you stay on LangChain. You swap the store, not the search call.

**One index, one embedding model** (Lab 1). If you change `text-embedding-3-small` for another model, you must embed the handbook again. The store will not catch that for you.

## What you should be able to explain

> "A vector store keeps chunk vectors and returns the closest few. Under the hood that is embed + cosine."

> "`from_documents` is index time. `retriever.invoke` is question time."

> "InMemoryVectorStore dies when the kernel restarts. Production uses Postgres, Chroma, Pinecone, or similar."

**Lab 6** takes these hits and asks a chat model to answer **only** from them.
